# 05. Regression Training & MLflow Tracking — EMIPredict AI

This notebook trains 4 regression algorithms (Linear Regression, Random Forest, Gradient Boosting, and XGBoost) to predict `max_monthly_emi` using the 48 scaled and engineered features. All experiments are logged to MLflow (hosted on DagsHub with fallback to local `mlruns/`).

In [ ]:
import os
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import mlflow
import mlflow.sklearn
import mlflow.xgboost
from mlflow.models.signature import infer_signature

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# MLflow Setup (DagsHub Remote with fallback to local mlruns)
try:
    from google.colab import userdata
    dagshub_uri = userdata.get('DAGSHUB_MLFLOW_URI') or userdata.get('MLFLOW_TRACKING_URI')
    dagshub_user = userdata.get('DAGSHUB_USERNAME') or userdata.get('MLFLOW_TRACKING_USERNAME')
    dagshub_token = userdata.get('DAGSHUB_TOKEN') or userdata.get('MLFLOW_TRACKING_PASSWORD')
    if dagshub_uri:
        os.environ['MLFLOW_TRACKING_URI'] = dagshub_uri
    if dagshub_user:
        os.environ['MLFLOW_TRACKING_USERNAME'] = dagshub_user
    if dagshub_token:
        os.environ['MLFLOW_TRACKING_PASSWORD'] = dagshub_token
except Exception:
    pass

tracking_uri = os.environ.get('MLFLOW_TRACKING_URI')
if not tracking_uri:
    mlruns_dir = '../mlruns' if os.path.exists('../mlruns') or os.path.exists('../notebooks') else 'mlruns'
    os.makedirs(mlruns_dir, exist_ok=True)
    tracking_uri = f"file:{os.path.abspath(mlruns_dir)}"

mlflow.set_tracking_uri(tracking_uri)
mlflow.set_experiment("emi_regression")
print(f"MLflow tracking configured at: {mlflow.get_tracking_uri()}")

In [ ]:
# Load Engineered Dataset (48 Features)
data_path = '../data/processed/engineered_dataset.csv' if os.path.exists('../data') else 'data/processed/engineered_dataset.csv'
df = pd.read_csv(data_path)

feature_cols = [
    'age', 'monthly_salary', 'years_of_employment', 'monthly_rent', 'family_size',
    'dependents', 'school_fees', 'college_fees', 'travel_expenses', 'groceries_utilities',
    'other_monthly_expenses', 'existing_loans', 'current_emi_amount', 'credit_score',
    'bank_balance', 'emergency_fund', 'requested_amount', 'requested_tenure',
    'debt_to_income_ratio', 'expense_to_income_ratio', 'affordability_ratio', 'risk_score',
    'salary_credit_interaction', 'surplus_to_requested_ratio',
    'gender_Female', 'gender_Male',
    'marital_status_Married', 'marital_status_Single',
    'education_Graduate', 'education_High School', 'education_Post Graduate', 'education_Professional',
    'employment_type_Government', 'employment_type_Private', 'employment_type_Self-employed',
    'company_type_Enterprise', 'company_type_Government', 'company_type_Local', 'company_type_MNC', 'company_type_Startup',
    'house_type_Family', 'house_type_Own', 'house_type_Rented',
    'emi_scenario_E-commerce Shopping', 'emi_scenario_Education', 'emi_scenario_Home Appliances', 'emi_scenario_Personal Loan', 'emi_scenario_Vehicle'
]

train_df = df[df['dataset_split'] == 'train']
val_df = df[df['dataset_split'] == 'val']

X_train, y_train = train_df[feature_cols].values, train_df['max_monthly_emi'].values
X_val, y_val = val_df[feature_cols].values, val_df['max_monthly_emi'].values

print(f"Train shape: {X_train.shape} (48 features), Validation shape: {X_val.shape}")

In [ ]:
# Model Candidates Definition
models = {
    "Linear Regression": LinearRegression(),
    "Random Forest Regressor": RandomForestRegressor(n_estimators=100, max_depth=12, random_state=42),
    "Gradient Boosting Regressor": GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, random_state=42),
    "XGBoost Regressor": XGBRegressor(n_estimators=100, max_depth=6, learning_rate=0.1, random_state=42)
}

# Training & Logging Loop
for name, model in models.items():
    with mlflow.start_run(run_name=name):
        print(f"Training {name}...")
        model.fit(X_train, y_train)
        
        y_pred = model.predict(X_val)
        
        mse = mean_squared_error(y_val, y_pred)
        rmse = np.sqrt(mse)
        mae = mean_absolute_error(y_val, y_pred)
        r2 = r2_score(y_val, y_pred)
        
        mlflow.log_params(model.get_params() if hasattr(model, 'get_params') else {})
        mlflow.log_metrics({
            "rmse": rmse,
            "mae": mae,
            "r2_score": r2
        })
        
        # Residuals Plot Artifact
        residuals = y_val - y_pred
        plt.figure(figsize=(6, 4))
        plt.scatter(y_pred, residuals, alpha=0.3, s=10)
        plt.axhline(0, color='red', linestyle='--')
        plt.title(f'Residuals vs Predicted - {name}')
        plt.xlabel('Predicted Max Monthly EMI')
        plt.ylabel('Residuals')
        res_path = f"residuals_{name.lower().replace(' ', '_')}.png"
        plt.tight_layout()
        plt.savefig(res_path)
        plt.close()
        mlflow.log_artifact(res_path)
        if os.path.exists(res_path):
            os.remove(res_path)
        
        # Log Model with Signature
        signature = infer_signature(X_val, y_pred)
        if "XGB" in name:
            mlflow.xgboost.log_model(model, artifact_path="model", signature=signature)
        else:
            mlflow.sklearn.log_model(model, artifact_path="model", signature=signature)
            
        print(f"Successfully logged {name} to MLflow (RMSE: {rmse:.2f}, R2: {r2:.4f})")